# 미니 GPT 밑바닥 구현

**30일 딥러닝 완전 정복 · Day 24** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- 셰익스피어 텍스트로 1천만 파라미터 문자 단위 GPT를 학습한다 (T4에서 약 10~15분)
- 탐욕 / 온도 / top-k / top-p 생성 전략을 비교한다
- 학습된 어텐션 헤드가 무엇을 보는지 시각화한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

## 1. 데이터

In [ ]:
import urllib.request, torch
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")
chars = sorted(set(text)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long); n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(len(text), "chars, vocab", V)

## 2. 모델 (강의 본문 코드와 동일 + 어텐션 저장 옵션)

In [ ]:
import math, time, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(1337)

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    d_model: int = 384
    dropout: float = 0.2

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head; self.p = cfg.dropout
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model); self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.save_attn = False; self.attn = None
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q, k, v = [t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2) for t in (q, k, v)]
        if self.save_attn:                      # 시각화용: 어텐션 가중치를 직접 계산해 저장
            att = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
            att = att.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1), float("-inf"))
            self.attn = att.softmax(-1).detach().cpu()
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, 4 * cfg.d_model), nn.GELU(), nn.Linear(4 * cfg.d_model, cfg.d_model), nn.Dropout(cfg.dropout))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.mlp = CausalSelfAttention(cfg), MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model); self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
        if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        logits = self.head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [ ]:
cfg = GPTConfig(vocab_size=V)
model = GPT(cfg).to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")
def get_batch(split, B=64):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - cfg.block_size - 1, (B,))
    x = torch.stack([d[i:i+cfg.block_size] for i in ix]); y = torch.stack([d[i+1:i+cfg.block_size+1] for i in ix])
    return x.to(device), y.to(device)
x, y = get_batch("train"); print("initial loss:", model(x, y)[1].item(), " expected ~", round(math.log(V), 3))

## 3. 학습

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
max_iters, warmup, lr_max, lr_min = 5000, 200, 1e-3, 1e-4
def lr_at(it):
    if it < warmup: return lr_max * (it + 1) / warmup
    r = (it - warmup) / (max_iters - warmup); return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * r))
@torch.no_grad()
def estimate_loss(iters=40):
    model.eval(); out = {}
    for s in ("train", "val"): out[s] = torch.stack([model(*get_batch(s))[1] for _ in range(iters)]).mean().item()
    model.train(); return out
amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))
t0 = time.time()
for it in range(max_iters + 1):
    for g in opt.param_groups: g["lr"] = lr_at(it)
    x, y = get_batch("train")
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    if it % 500 == 0:
        l = estimate_loss(); print(f"iter {it:5d}  train {l['train']:.3f}  val {l['val']:.3f}  ({time.time()-t0:.0f}s)")

## 4. 생성 전략 비교

In [ ]:
@torch.no_grad()
def generate(idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None, greedy=False):
    model.eval()
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -cfg.block_size:])
        logits = logits[:, -1, :]
        if greedy:
            nxt = logits.argmax(-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, -1)
            if top_p is not None:
                sp, si = torch.sort(probs, descending=True); cum = sp.cumsum(-1)
                sp[cum - sp > top_p] = 0; probs = torch.zeros_like(probs).scatter(-1, si, sp); probs /= probs.sum(-1, keepdim=True)
            nxt = torch.multinomial(probs, 1)
        idx = torch.cat([idx, nxt], 1)
    model.train(); return decode(idx[0].tolist())

start = torch.tensor([encode("ROMEO:\n")], device=device)
for name, kw in [("greedy", dict(greedy=True)), ("temp 0.5", dict(temperature=0.5)), ("temp 1.0 + top-k 40", dict(temperature=1.0, top_k=40)), ("temp 1.0 + top-p 0.9", dict(temperature=1.0, top_p=0.9))]:
    print(f"\n========== {name} =========="); print(generate(start, 300, **kw))

## 5. 어텐션 시각화
마지막 층의 각 헤드가 어디를 보는지 히트맵으로 그립니다. 바로 앞 글자를 보는 헤드, 단어 시작이나 줄바꿈을 보는 헤드 등이 있는지 찾아보세요.

In [ ]:
import matplotlib.pyplot as plt
s = "ROMEO:\nBut soft, what light through yonder window breaks?"
ids = torch.tensor([encode(s)], device=device)
layer = model.blocks[-1].attn; layer.save_attn = True
model.eval(); model(ids); layer.save_attn = False; model.train()
A = layer.attn[0]                                   # (heads, T, T)
fig, axs = plt.subplots(2, 3, figsize=(15, 9))
for h, ax in enumerate(axs.ravel()):
    ax.imshow(A[h], cmap="viridis"); ax.set_title(f"head {h}")
    ax.set_xticks(range(len(s))); ax.set_xticklabels(list(s.replace("\n", "⏎")), fontsize=6)
    ax.set_yticks(range(len(s))); ax.set_yticklabels(list(s.replace("\n", "⏎")), fontsize=6)
plt.tight_layout(); plt.show()

## 과제
- 검증 손실 1.6 이하를 달성했나요?
- `n_layer=2`로 다시 학습해 생성 품질을 비교하세요.
- 생성 전략별 결과를 학습 노트에 붙이세요.